## CSCE 676 Project Checkpoint 2: Research Question Formation

# Section A: Project Scope

**Dataset: 9000+ Movies : IMDb and Bechdel (Kaggle)**

https://www.kaggle.com/datasets/nliabzd/movies-imdb-and-bechdel-information

Course Topic Alignment: Frequent Itemsets, Association Rules, Anomaly Detection

Beyond Course Techniques: Causal Inference, Quantile Regression

Dataset Size: 9000+ movies with Bechdel scores

Data Types: Text (Title, Genre), Numeric (Year, Runtime, Vote Count), IMDb ID, Bechdel Test ID, Bechdel Rating, IMDb Rating

EDA Findings: The dataset exhibits genre dominance (Drama and Comedy are most frequent). There is a slight inverse relationship between high Bechdel scores (3) and median IMDb ratings. A significant portion of the dataset has full "baskets" of up to three genres, making it apt for itemset mining.

# Section B: Research Question Definition

**Research Question 1: How do varying support and confidence thresholds impact the discovery of rare, high-lift genre itemsets that predict Bechdel Test failures?**

Data Mining Task Type: Frequent Itemset & Association Rule Mining

Relevant Algorithm(s): Apriori & FP-Growth

Evaluation Criteria: Support, confidence, lift, and execution time across different thresholds

Motivation: The dataset is heavily skewed toward Drama/Comedy. If we use standard thresholds, we might miss critical patterns in sparse, niche genres (like Film Noir or Westerns). This tests the limits of the algorithm on imbalanced categorical data.

**Research Question 2:  How do density-based anomaly detection methods compare to tree-based isolation methods when identifying representation outliers in highly dimensional data?**

Data Mining Task Type: Anomaly Detection

Relevant Algorithm(s): DBSCAN & Isolation Forest

Evaluation Criteria: Silhouette score, Sensitivity to contamination rate parameters and interpretability of anomalies

Motivation: Anomaly detection is complicated when mixing categorical (Genre) and continuous (Budget, Year) data. Comparing how different algorithms define an "outlier" in this specific context would lead to interesting conclusions.

**Research Question 3: How do causal inference methods improve the estimate of the Bechdel Test's impact on ROI, and how does quantile regression capture varying effects across tiers of financial success?**

Data Mining Task Type: Causal Inference & Quantile Regression

Relevant Algorithm(s): Propensity Score Matching, Ordinary Least Squares, Quantile Regression

Evaluation Criteria: Average Treatment Effect on the Treated (ATT) for causal impact and comparison of regression coefficients at the 10th, 50th, and 90th percentiles for the quantile analysis

Motivation: Standard correlation is insufficient because financial success is heavily influenced by budget and genre. Causal inference is needed to isolate whether passing the test actually increases ROI.

# Section C: Motivation and Feasibility


Motivation: The EDA revealed that while many films pass the Bechdel test, there's a slight dip in median user ratings and female-representation likely varies by genre. Standard correlations can't verify that female-led films are penalized financially or critically.

Non-triviality: Standard course techniques like Association Rules will find basic patterns, but they ignore the complex variables (Budget and Year) that are necessary for this dataset.

Feasibility: Python libraries like mlxtend make Apriori and FP-Growth  implementable on categorical data. The EDA confirmed that most movies have 1-3 genres, creating natural "baskets.". The dataset's tabular nature makes it highly compatible with libraries like scikit-learn (for DBSCAN & Isolation Forests) and statsmodels (for Quantile Regression and Causal Inference). The dataset size (~9,000 rows) is large enough for analysis but can run quickly on a local machine.

Risks: Sub-genres might lack the support needed for meaningful association rules. Also, mixing continuous and categorical data can complicate distance-based anomaly detection

# Section D: Methodological Planning

Course algorithms: Apriori, FP-Growth (for frequent itemsets), DBSCAN, Isolation Forest (for anomaly detection).

External algorithms: Propensity Score Matching (for causal inference), Quantile Regression (for distributional effects)

Evaluation Metrics:

- RQ1: Support, Confidence, Lift, and algorithm execution time
- RQ2: Silhouette score and sensitivity to the contamination parameter
- RQ3: Average Treatment Effect on the Treated, p-values, quantile coefficients

Baselines:

- RQ1: The dataset's overall baseline probability of a movie failing the Bechdel test which "Lift" will be measured against
- RQ2: Z-score filtering on numerical columns (Budget, Rating) to compare against isolation techniques
- RQ3: Ordinary Least Squares regression modeling ROI based on Bechdel scores to show the limitations of using mean-based models

**RQ-to-Method Mapping Table:**
| Research Question | Task Type | Course vs. External | Algorithm(s) |
| :--- | :--- | :--- | :--- |
| RQ1 | Association Rules | Course | Apriori, FP-Growth |
| RQ2| Anomaly Detection | Course | DBSCAN, Isolation Forest |
| RQ3| Causal Inference & Regression | External | PSM, Quantile Regression |

In [6]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# ==========================================
# RQ1
# ==========================================

# Load dataset into DataFrame
df = pd.read_csv('/content/Bechdel_IMDB_Merge0524.csv')

genre_data = df[['genre1', 'genre2', 'genre3']].dropna(how='all')
transactions = genre_data.values.tolist()

transactions = [[str(genre) for genre in movie if str(genre) != 'nan'] for movie in transactions]

# 2. One-Hot Encode the transactions: Apriori requires a boolean matrix where columns are items and rows are transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_genres_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# 3. Initial Method Run: Apriori Algorithm
# Use a low minimum support (0.05 or 5%) to see if the algorithm can find itemsets other than 'Drama' and 'Comedy'
frequent_itemsets = apriori(df_genres_encoded, min_support=0.05, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

print("Initial Apriori Run Feasibility Check (Top 5 Rules by Lift):")
display(rules.sort_values('lift', ascending=False).head(5))

Initial Apriori Run Feasibility Check (Top 5 Rules by Lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Action),(Adventure),0.196337,0.174213,0.086026,0.438155,2.515057,1.0,0.051822,1.469778,0.749561,0.302351,0.319625,0.465977
1,(Adventure),(Action),0.174213,0.196337,0.086026,0.493798,2.515057,1.0,0.051822,1.587634,0.729479,0.302351,0.370132,0.465977
2,(Biography),(Drama),0.058037,0.570694,0.051348,0.884752,1.550310,1.0,0.018227,3.725059,0.376838,0.088932,0.731548,0.487363
3,(Drama),(Biography),0.570694,0.058037,0.051348,0.089975,1.550310,1.0,0.018227,1.035096,0.826840,0.088932,0.033906,0.487363
4,(Romance),(Comedy),0.179667,0.372813,0.090657,0.504582,1.353444,1.0,0.023674,1.265974,0.318339,0.196301,0.210095,0.373875


In [10]:
# ==========================================
# RQ2
# ==========================================
from sklearn.ensemble import IsolationForest
import numpy as np

# Running a basic Isolation Forest to ensure we can handle the mixed numerical/categorical data before tuning hyperparameters

df_rq2 = df[['imdbAverageRating', 'runtimeMinutes']].copy()
df_rq2.replace('\\N', np.nan, inplace=True)
df_rq2 = df_rq2.dropna()

df_rq2['imdbAverageRating'] = pd.to_numeric(df_rq2['imdbAverageRating'])
df_rq2['runtimeMinutes'] = pd.to_numeric(df_rq2['runtimeMinutes'])

# Use a default contamination rate of 0.05 just to test if the model can successfully isolate 5% of the data as outliers.
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_rq2['anomaly_label'] = iso_forest.fit_predict(df_rq2[['imdbAverageRating', 'runtimeMinutes']])

print("RQ2 Feasibility: Isolation Forest found this many anomalies (-1) vs normal (1):")
print(df_rq2['anomaly_label'].value_counts())

RQ2 Feasibility: Isolation Forest found this many anomalies (-1) vs normal (1):
anomaly_label
 1    9224
-1     486
Name: count, dtype: int64


In [11]:
# ==========================================
# RQ3
# ==========================================
import statsmodels.formula.api as smf

# Testing the 'statsmodels' to ensure dataset has enough variance to do distributional analysis

df_rq3 = df[['imdbAverageRating', 'bechdelRating']].dropna()

# Start with q=0.5 (median) as baseline test before exploring the 10th and 90th percentiles
try:
    mod = smf.quantreg('imdbAverageRating ~ bechdelRating', df_rq3)
    res = mod.fit(q=0.5)

    print("RQ3 Feasibility: Quantile Regression (q=0.5) successfully ran.")
    print(res.summary().tables[1])
except Exception as e:
    print(f"Error running Quantile Regression: {e}")

RQ3 Feasibility: Quantile Regression (q=0.5) successfully ran.
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         6.9000      0.027    259.861      0.000       6.848       6.952
bechdelRating    -0.1000      0.011     -9.098      0.000      -0.122      -0.078


On my honor, I declare the following resources:
1. Collaborators:
- NA

2. Web Sources:
- Bechdel Test Movie List https://bechdeltest.com/

3. AI Tools:
- Gemini: Prompted to rephrase RQs from original draft text. Used to generate code for initial method testing for all RQs.

4. Citations:
- 9000+ Movies : IMDb and Bechdel https://www.kaggle.com/datasets/nliabzd/movies-imdb-and-bechdel-information
